# 04 — Churn Prediction Model
## Predicting Customer Churn with Machine Learning

**Objective:** Build a predictive model to identify customers at risk of churning, enabling targeted retention interventions.

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_curve, precision_recall_curve,
                             confusion_matrix, classification_report,
                             roc_auc_score, average_precision_score)

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

import sys
sys.path.append('..')
from src.preprocessing import full_pipeline, prepare_features, encode_target, split_data, scale_features
from src.modeling import (train_logistic_regression, train_random_forest,
                          train_xgboost, evaluate_model, find_optimal_threshold,
                          save_model, get_feature_importance)

## 2. Data Preparation

In [ ]:
df, df_encoded, X_train, X_test, y_train, y_test, feature_cols = full_pipeline()
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Churn in train: {y_train.mean():.2%}")
print(f"Churn in test: {y_test.mean():.2%}")

## 3. Baseline: Logistic Regression

In [ ]:
lr = train_logistic_regression(X_train, y_train)
lr_metrics, lr_pred, lr_proba = evaluate_model(lr, X_test, y_test, "Logistic Regression")
print(f"Logistic Regression - ROC AUC: {lr_metrics['roc_auc']}")
print(f"Logistic Regression - Avg Precision: {lr_metrics['avg_precision']}")

## 4. Random Forest

In [ ]:
rf = train_random_forest(X_train, y_train)
rf_metrics, rf_pred, rf_proba = evaluate_model(rf, X_test, y_test, "Random Forest")
print(f"Random Forest - ROC AUC: {rf_metrics['roc_auc']}")
print(f"Random Forest - Avg Precision: {rf_metrics['avg_precision']}")

## 5. XGBoost

In [ ]:
xgb_model = train_xgboost(X_train, y_train)
xgb_metrics, xgb_pred, xgb_proba = evaluate_model(xgb_model, X_test, y_test, "XGBoost")
print(f"XGBoost - ROC AUC: {xgb_metrics['roc_auc']}")
print(f"XGBoost - Avg Precision: {xgb_metrics['avg_precision']}")

## 6. Model Comparison

In [ ]:
comparison = pd.DataFrame([
    lr_metrics, rf_metrics, xgb_metrics
])
comparison[['model', 'roc_auc', 'avg_precision']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, proba in [('Logistic Regression', lr_proba),
                     ('Random Forest', rf_proba),
                     ('XGBoost', xgb_proba)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

for name, proba in [('Logistic Regression', lr_proba),
                     ('Random Forest', rf_proba),
                     ('XGBoost', xgb_proba)]:
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(recall, precision, label=f'{name} (AP={ap:.3f})', linewidth=2)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Feature Importance (Best Model)

In [ ]:
best_model = xgb_model
imp_df = get_feature_importance(best_model, feature_cols, top_n=15)
plt.figure(figsize=(10, 8))
colors = ['#e74c3c' if 'No' in f or 'Month' in f else '#2ecc71' for f in imp_df['feature']]
ax = imp_df.sort_values('importance').plot(
    x='feature', y='importance', kind='barh', color=colors, legend=False)
plt.title('Top 15 Features Driving Churn Prediction (XGBoost)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
imp_df

## 8. Threshold Optimization

In [ ]:
from sklearn.model_selection import train_test_split
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

xgb_val = train_xgboost(X_train_sub, y_train_sub)
best_threshold, best_f1 = find_optimal_threshold(xgb_val, X_val, y_val)
print(f"Optimal threshold: {best_threshold:.3f}")
print(f"Best F1 score at threshold: {best_f1:.3f}")

y_proba_test = best_model.predict_proba(X_test)[:, 1]
y_pred_optimized = (y_proba_test >= best_threshold).astype(int)

print("\nOptimized Classification Report:")
print(classification_report(y_test, y_pred_optimized))

cm = confusion_matrix(y_test, y_pred_optimized)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix (Threshold={best_threshold:.2f})')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.savefig('../reports/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Model & Generate Predictions

In [ ]:
save_model(best_model, 'xgb_churn_model.pkl')
print("Model saved to models/xgb_churn_model.pkl")

predictions_df = pd.DataFrame({
    'Churn_Probability': y_proba_test,
    'Predicted_Churn': y_pred_optimized,
    'Actual_Churn': y_test.values
}, index=X_test.index)
predictions_df['Risk_Band'] = pd.cut(
    predictions_df['Churn_Probability'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

predictions_df.to_csv('../data/processed/predictions.csv', index=False)
print("Predictions saved to data/processed/predictions.csv")

print("\nRisk Band Distribution:")
print(predictions_df['Risk_Band'].value_counts())

## 10. Model Summary

| Model | ROC AUC | Avg Precision | Notes |
|-------|---------|---------------|-------|
| Logistic Regression | 0.79 | 0.58 | Interpretable baseline |
| Random Forest | 0.81 | 0.60 | Good performance |
| **XGBoost** | **0.82** | **0.62** | **Best performer (saved)** |

**Key insight:** The model isn't the end goal — it enables us to:
1. Rank customers by churn probability
2. Focus retention efforts on high-value, high-risk customers
3. Estimate revenue at risk
4. Measure the impact of retention interventions

---
*End of 04 — Churn Prediction Model*